# From neural networks to deep leaning

In this next bit we will finally leave our faithful friend `sklearn` and move to a package specifically for deep neural networks. 

When we talk about "deep" netwoks, we simply mean neural networks that have more than one hidden layer. 

These deep neural networks have experimentally been found to be extrememly good for data with complex structures like

- natural language (LLMs)
- images
- video
- audio
- protein structures (alphafold)

The nature of NN training means that they are also useful for certain problems that don't really fall into our supervised/unsupervised paradigm e.g. reinforcement learning. 

As we have seen already, on simple problems like our penguin dataset, NNs aren't particularly interesting. 

## Pick a framework
We could do this with our from-scratch `pandas`/`numpy` based code, but it would involve a lot of copy/pasting and be error prone and very slow. We could `sklearn`, but it's as it's a general-purpose ML framework it isn't optimised for neural networks. So we will switch to a different library that is.  

If we want to use NNs for real world tasks, we will need to switch to a dedicated library. `sklearn` by design has to support a lot of different ML models, so things can't be optimized for the way that NNs work. 

There are a bunch of different libraries that we can use in Python...

| Framework         | Primary Language(s)      | Targeted Problems / Strengths |
|-------------------|--------------------------|--------------------------------|
| **TensorFlow**    | Python, C++, JavaScript | General-purpose DL (CV, NLP, RL, speech); production & deployment at scale |
| **PyTorch**       | Python, C++             | Research-focused DL (CV, NLP, generative models); strong flexibility & dynamic graphs |
| **Keras**         | Python                  | High-level interface for rapid prototyping of CV, NLP, structured data; ease of use |
| **JAX**           | Python                  | High-performance ML research; auto-differentiation, large-scale training (esp. scientific ML, LLMs) |


They all use more or less the same set of concepts that we have been talking about in the from-scratch and sklearn code:

- neurons
- layers
- weight and biases
- activation functions
- gradient calculations
- backpropagation
- convergence
- loss
- epochs
- batches
- optimisers
- etc. etc.

So we will pick one that is widely used, reasonably easy to get working, and performant: pytorch. But it will be relatively easy to pick up another if necessary!

`pytorch` has a few features that make it good for us:

- uses Python classes in a pretty straightforward way
- uses tensors, which are familiar as long as you have used numpy a bit
- you write explicit training loops, which makes the training process clearer
- easy (relatively speaking) to switch to GPU if you want
- has a bunch of ecosystem packages for e.g vision, monitoring

If you have **never** used Python classes, or numpy, some parts of the code might be mysterious, but you can always go back and fill in the missing pieces later. 

What we will notice vs. sklearn:

- one model type rather than many (but massive variation in paramters, architecture)
- more ceremony around loading data to cope with e.g. massive image datasets
- training is more of a process rather than one API call

In practice `pytorch` code will feel like a cross between our from-scratch numpy/pandas code and sklearn NN code.

Let us remember not to get too excited about deep learning. Although they are the new hotness, and theoretically univeral function-learning machines, classical ML models are still

- higher scoring on tabular data
- more interpretable
- much less computationally intensive
- much less data intensive

Colab includes GPU-enabled PyTorch and torchvision. After selecting Runtime > Change runtime type > GPU, run the next cell to check availability. Do not run a CPU-only PyTorch installer here.

In [ ]:
import torch, torchvision
print("torch", torch.__version__, "torchvision", torchvision.__version__)
print("CUDA available?", torch.cuda.is_available())


## Setting up a regression model in `pytorch`

OK, let's see what this will look like. Compared to previous code:

- we will take a bunch of shortcuts with sklearn to do things we have previously written out explicitly with pandas
- we will drop meaningful variable names like species, bill_length etc. and just refer to X and y values
- we will no longer refer to individual colums, but will take our entire table of features and represent it as an array/vector/tensor

### Getting the data ready

We will do this in a few steps. First, 

In [ ]:
import pandas as pd, numpy as np, torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Variable naming convention notes:
#   - X* : feature matrices (capital X is standard for design matrix)
#   - y* : target vectors (lowercase y for labels)
#   - *_str : string labels before encoding
#   - *_np : numpy arrays (after preprocessing with sklearn)
#   - *_t : torch tensors (converted from numpy, ready for DataLoader)
#   - *_ds : TensorDataset objects
#   - *_loader : DataLoader objects wrapping datasets
#   - *_cols : list of column names (numeric or categorical)

# load the csv as normal
df = pd.read_csv('https://gist.githubusercontent.com/slopp/ce3b90b9168f2f921784de84fa445651/raw/4ecf3041f0ed4913e7c230758733948bc561f434/penguins.csv').dropna()

# get rid of rows/columns we don't want
df = df.dropna()  # drop rows with missing data to simplify the model

# we know that year is not a useful feature
# island will make the problem uninteresting, so remove it
X = df.drop(columns=["species", "year", "island", "rowid"], errors="ignore") 

# we will turn the species names into integer labels
# we could easily do this with pandas, but to be lazy we will let sklearn handle it
label_enc = LabelEncoder()
y = label_enc.fit_transform(df["species"].values)
classes = label_enc.classes_

Let's see what we have at this stage:

In [ ]:
X.head()

In [ ]:
y, classes

Next we will do our feature engineering. We need to rescale our numeric columns and encode our categorical columns (just sex in this case). We have done all this previously with pandas, so we will use sklearn.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


# just write out a list of numeric/cat columns
num_cols = ["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]
cat_cols = ["sex"] 

# ColumnTransformer: apply scaler to numeric cols and encoder to cat cols.
pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

# we will split into train, test and validation sets using tts as we have seen many times
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Remember; the scaling is only fit on the training set, the others just transform
pre.fit(X_train)
X_train_np = pre.transform(X_train)
X_val_np   = pre.transform(X_val)
X_test_np  = pre.transform(X_test)


Now hopefully we have three datasets - train/test/validate - with different sizes but the same number of columns:

In [ ]:
y_train

In [ ]:
X_train_np.shape, X_test_np.shape, X_val_np.shape

Now is a good time to get comfortable thinking in terms of n-dimensional vectors if you are not already :) A difference we will notice from our from-scrach code is that we will not generally refer to column names any more.

Note that we have removed column names and are now working with an array of floats:

In [ ]:
# remember one-hot encoding uses n-1 values, so the species is represented by the two final 0/1 columns
# all other columns are scaled to -1,1
X_train_np

The corresponding `y` values are just lists of labels:

In [ ]:
# notice that tts has shuffled the data, they are no longer in species order
y_train

For the differentiation magic to work, the np arrays have to be turned into tensors:

In [ ]:
# convert the numpy arrays to tensors
# the features will all be floating point numbers
X_train_t = torch.tensor(X_train_np.toarray() if hasattr(X_train_np, "toarray") else X_train_np, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_np.toarray() if hasattr(X_val_np, "toarray") else X_val_np, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_np.toarray() if hasattr(X_test_np, "toarray") else X_test_np, dtype=torch.float32)

# the labels will be integers so store them as type long
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t   = torch.tensor(y_val, dtype=torch.long)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

For our purposes, these are very similar to np arrays (which in turn are very similar to pd dataframes) - they are just containers for the feature vectors. 

In [ ]:
X_test_t

### Datasets and Data loaders

Now comes a bit that is more work than we have done before. Because `pytorch` is intended to be used with datasets that are too big to fit in memory, there are special classes to represent datasets, and special classes to handle loading them bit by bit. This is a good example of a feature that is annoying for small/simple datasets, but becomes very useful for big/complex ones. 

This is mostly just wrapping, the only interesting thing we do here is set the batch size (how many datapoints to evaluate before updating the weights when training)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t, y_val_t)
test_ds  = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=128)
test_loader  = DataLoader(test_ds, batch_size=128)

train_loader

A few variables will be useful later:

In [ ]:
# number of features
input_dim = X_train_t.shape[1]

# number of classes
num_classes = len(classes)

input_dim, num_classes

This is a lot of code just to get the dataset ready! But much of it we have had to do anyway for other model types as well.

### Setting up the training loop

Now to describe the network we want to build. In the sklearn world we did something like this:

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(
    
    # one hidden layer with three neurons, use tanh as the activation function
    hidden_layer_sizes=(3,),
    activation='tanh', 

    # update the weights as we have been doing it; one update for the whole dataset each epoch
    # don't change the learning rate, no momentum, no regularization
    solver='sgd', 
    batch_size=X.shape[0],        
    learning_rate_init=1,
    momentum=0.0,
    alpha=0.0,  
)

Here we set the layer sizes and activation function. But there are a few drawbacks. What if we want different activation functions for different layers? Or some sort of complicated non-fully-connected layer structure? Also note that we are mixing up a bunch of network architecture arguments (layers, activation) with a bunch of details about how the model will be trained. Much better to separate these. 

So this is how it works in `pytorch`:

In [ ]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, in_features, hidden=32, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.Tanh(),
            nn.Linear(hidden, num_classes)  # logits
        )
    def forward(self, x):
        return self.net(x)

The main bit is where we define the network `self.net`. Think of it as defining the layers of weights **between** neurons, rather than the neurons themselves. Here we say that we want:

- an input layer (`nn.Linear`) with a number of neurons equal to the number of features, just like we had in our from-scratch code, and an output number of whatever number of neurons we want in the hidden layer. We could just hard code these like `nn.Linear(6, 42)` but we will probably want to try different numbers, and we need to make sure that they match in both sets of weights, so we make both the input and output size variables.

- next we have the activation function for the neurons - just like before we can choose from various ones

- finally the weights leading to the output neurons. The number of inputs here needs to be the size of the hidden layer, and the number of outputs is the number of classes. 

Now to create an instance of this class. We could hard code the dimensions here:

In [ ]:
model = MLP(6, hidden=32, num_classes=3)
model

or we could grab the values from the dataset itself:

In [ ]:
model = MLP(input_dim, hidden=32, num_classes=num_classes)
model

Notice that when we evaluate the model variable we get a nice summary of the structure. Before we do anything else, let's check that we can run it with random weights. The random starting weights are assigned when we create the instance. So just like with our from-scratch or sklearn models, we can pick some random feature values and plug them into the model:

In [ ]:
# just like sklearn, this expects a list of lists
# so we have to package it up even if we have just one input to process
model(
    torch.tensor([[-2.2181, -1.0634, -1.5775,  0.5323, 0,  1]])
)

And we get out a list of output weights for the three output neurons, one per species. Normally we will run these through a softmax function to get class probabilities:

In [ ]:
logits = model(
    torch.tensor([[-2.2181, -1.0634, -1.5775,  0.5323, 0,  1]])
)

torch.softmax(
    logits, 
    dim=1
)

Or if we want we can just get the output with the highest score and treat that as the prediction:

In [ ]:
torch.argmax(logits, dim=1)

In [ ]:
classes

Now to put together the training loop. This will be familiar from our from-scratch code, at least in terms of the order of operations. 

In [ ]:
import torch, numpy as np
from torch import nn
from torch.optim import Adam, SGD

# copied from before
model = MLP(input_dim, hidden=32, num_classes=num_classes)

# convenience class to calculate the loss; for regression this would be something like SSE
criterion = nn.CrossEntropyLoss()

# which algorithm we will use to adjust the weights, has a learning rate just like before
optimizer = SGD(model.parameters(), lr=1e-2)

# make sure the model is in training mode
model.train()

# Training loop helper function
# remember one epoch is one complete run through all the data points


def run_epoch(data_loader):
    """Run a single training epoch and return average loss."""
    
    total_loss, total_count = 0.0, 0

    # each iteration of this loop is one mini-batch
    for features, labels in data_loader:
        
        # Clear gradients from the previous batch
        optimizer.zero_grad()
        
        # Forward pass: compute raw outputs (logits) from the model
        logits = model(features)
        
        # Compute loss comparing predictions (logits) with true labels
        loss = criterion(logits, labels)
        
        # Backward pass: calculate gradients of loss w.r.t. model parameters
        loss.backward()
        
        # Update model parameters using optimizer and accumulated gradients
        optimizer.step()

        # Track total loss (scaled by batch size) for reporting
        total_loss += loss.item() * features.size(0)
        # Count number of samples seen so far
        total_count += features.size(0)

    average_loss = total_loss / total_count
    return average_loss

loss_scores = []

# Fixed number of epochs
num_epochs = 500 
for epoch in range(1, num_epochs + 1):
    train_loss = run_epoch(train_loader)
    loss_scores.append(train_loss)

    # print scores every 10 epochs
    if epoch % 50 == 0:
        print(f"Epoch {epoch:03d} | train loss {train_loss:.4f}")

In [ ]:
import seaborn as sns
sns.relplot(
    loss_scores,
    kind='line'
)

Each of these lines corresponds more or less to a bit from our from-scratch training code. The `pytorch` code is comparatively very compact since all the calculations for all layers happen in  `model(features)` and `loss.backward(), optimizer.step()`. This is where all the matrix algebra happens that took a lot of code when we were using `@`.

Notice that unlike sklearn, pytorch does not automatically store the loss/accuracy internally, so we need to store it explicitly. 

In real world use, there are a couple of tweaks we will often make to the training code. Firstly, we will want to know not just the loss for the training data, but for the test data too, so we will do an extra step after each epoch to see how the model does on the validation data:

In [ ]:
import torch, numpy as np
from torch import nn
from torch.optim import Adam


model = MLP(input_dim, hidden=32, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Training loop helper
def run_epoch(data_loader, is_training=True):
    if is_training:
        model.train()
    else:
        model.eval()

    total_loss, total_correct, total_count = 0.0, 0, 0
    for features, labels in data_loader:
        if is_training:
            optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        if is_training:
            loss.backward()
            optimizer.step()

        predictions = logits.argmax(dim=1)
        total_loss += loss.item() * features.size(0)
        total_correct += (predictions == labels).sum().item()
        total_count += features.size(0)

    avg_loss = total_loss / total_count
    avg_accuracy = total_correct / total_count
    return avg_loss, avg_accuracy

# Run for a fixed number of epochs (no early stopping)
results = []
num_epochs = 5000
for epoch in range(num_epochs):
    train_loss, train_accuracy = run_epoch(train_loader, is_training=True)
    val_loss, val_accuracy = run_epoch(val_loader, is_training=False)
    results.append((epoch, train_loss, train_accuracy, val_loss, val_accuracy))
    if epoch % 50 == 0:
        print(f"Epoch {epoch:03d} | train loss {train_loss:.4f} acc {train_accuracy:.3f} | val loss {val_loss:.4f} acc {val_accuracy:.3f}")

# Save final model after training
torch.save({"model": model.state_dict(), "classes": classes}, "penguins_final.pt")


In [ ]:
results = pd.DataFrame(results, columns=['epoch', 'train_loss', 'train_accuracy', 'val_loss', 'val_accuracy'])

sns.relplot(
    results.melt(id_vars=['epoch']),
    x='epoch',
    y='value',
    col='variable',
    kind='line',
    facet_kws={'sharey': False}
)

This dataset is very small, so don't be surprised if by chance we end up with a model that actually performs better on the validation dataset than the training one. Notice that the loss can keep going down even while the accuracy stays the same (it can't get higher than one) - think of this as the model not only getting the correct species prediction, but being more confident about it. 

The second tweak we will often make is to include some logic to finsh the training early once the model is not improving any more:



In [ ]:
import torch, numpy as np
from torch import nn
from torch.optim import Adam

model = MLP(input_dim, hidden=3, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=1e-2)

# Training loop helper
def run_epoch(data_loader, is_training=True):
    if is_training:
        model.train()
    else:
        model.eval()

    total_loss, total_correct, total_count = 0.0, 0, 0
    for features, labels in data_loader:
        if is_training:
            optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        if is_training:
            loss.backward()
            optimizer.step()

        predictions = logits.argmax(dim=1)
        total_loss += loss.item() * features.size(0)
        total_correct += (predictions == labels).sum().item()
        total_count += features.size(0)

    avg_loss = total_loss / total_count
    avg_accuracy = total_correct / total_count
    return avg_loss, avg_accuracy

# allow 10 epochs with no loss improvement on the validation set before calling it a day
best_val_loss, patience, bad_epochs = np.inf, 10, 0
max_epochs = 5000
results = []

for epoch in range(max_epochs):
    train_loss, train_accuracy = run_epoch(train_loader, is_training=True)
    val_loss, val_accuracy = run_epoch(val_loader, is_training=False)
    results.append((epoch, train_loss, train_accuracy, val_loss, val_accuracy))
    if epoch % 50 == 0:
        print(f"Epoch {epoch:03d} | train loss {train_loss:.4f} acc {train_accuracy:.3f} | val loss {val_loss:.4f} acc {val_accuracy:.3f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss, bad_epochs = val_loss, 0
        torch.save({"model": model.state_dict(), "classes": classes}, "penguins_best.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("Early stopping.")
            break


In [ ]:
results = pd.DataFrame(results, columns=['epoch', 'train_loss', 'train_accuracy', 'val_loss', 'val_accuracy'])

sns.relplot(
    results.melt(id_vars=['epoch']),
    x='epoch',
    y='value',
    col='variable',
    kind='line',
    facet_kws={'sharey': False}
)

You will notice one more thing we have not had before; saving the model weights to a file. Since training might take a long time, we will not want to repeat it every time we want to use the model! Something like this will load the weights:

In [ ]:
# pass in the file
checkpoint = torch.load("penguins_best.pt", weights_only=False)

# take a look at the weights
checkpoint

Notice that it also stores the original class labels; if we lose track of these then we will not know which output label from the model refers to which original species, so it will not be very useful.

Now we can plug those weights back into an instance of the model:

In [ ]:
# Retrieve number of classes and rebuild model with the same dimensions
classes = checkpoint["classes"]
input_dim = 6  # must match the feature dimension used during training

model = MLP(in_features=input_dim, hidden=3, num_classes=len(classes))
model.load_state_dict(checkpoint["model"])
model.eval()  # set to evaluation mode

print("Loaded model with classes:", classes)

Now we can use the model for inference

In [ ]:
# show raw output neuron activations
model(X_test_t)

In [ ]:
# show actual predictions
model(X_test_t).argmax(1)

If we have some dataset and we construct a list of the true and predicted classes:

In [ ]:
predicted_classes = model(X_test_t).argmax(1)
true_classes = y_test_t

predicted_classes, true_classes

Then we can conveniently get metrics:

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(true_classes, predicted_classes, target_names=list(classes)))

In [ ]:
confusion_matrix(true_classes, predicted_classes)

## Image classification example

Where neural networks really have good performance is with datasets that have a complex structure. We will walk through a simple image classification example using 10 images from each of 12 plant species rather than the much larger original collection.

These images are a teaching subset of the [Plant Seedlings Dataset](https://vision.eng.au.dk/plant-seedlings-dataset/) by Giselsson and colleagues ([dataset paper](https://arxiv.org/abs/1711.05458)). The publisher requires citation and distributes the images under CC BY-SA; the download includes attribution and license details. This tiny sample is for learning the mechanics, not for reporting a reliable classification benchmark.

The next cell downloads a 44 MB archive once, checks its SHA-256 hash, and extracts a `plant_seedlings_120/` folder alongside this notebook. If that folder already exists locally, it uses it. It will **not** use the old full-size `plants/` folder. Colab's runtime storage is temporary, so a fresh runtime will download it again.


In [ ]:
from pathlib import Path
from hashlib import sha256
from urllib.request import urlretrieve
from zipfile import ZipFile

archive_url = "https://github.com/mojones/earth-science-ml-pytorch-colab/releases/download/v1.0/plant-seedlings-120.zip"
archive_hash = "ac87db8c3f5447208702e3ec11aa9cfc3b199eae03d38483c7a69150b6967ec1"
# Use the bundled local ZIP if it exists; otherwise download to this folder.
archive_path = Path("assets/plant-seedlings-120.zip")
if not archive_path.exists():
    archive_path = Path("plant-seedlings-120.zip")
data_dir = Path("plant_seedlings_120")

if not data_dir.exists():
    if not archive_path.exists():
        urlretrieve(archive_url, archive_path)
    actual_hash = sha256(archive_path.read_bytes()).hexdigest()
    if actual_hash != archive_hash:
        raise ValueError("Image archive hash mismatch. Delete the ZIP and try again.")
    with ZipFile(archive_path) as archive:
        archive.extractall(".")

species_folders = sorted(p for p in data_dir.iterdir() if p.is_dir())
image_count = sum(len(list(folder.glob("*.png"))) for folder in species_folders)
assert len(species_folders) == 12 and image_count == 120
print(image_count, "images in", len(species_folders), "species folders")
print([folder.name for folder in species_folders])


In [ ]:
print(sorted(p.name for p in (data_dir / "Shepherds Purse").glob("*.png")))


In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# brittle Python code, do not trust for real world

# Loop through each species folder and display up to 3 images
for species in os.listdir(data_dir) :
    folder = os.path.join(data_dir, species)
    img_files = os.listdir(folder)[:3]

    images = []
    print(img_files)
    for fname in img_files:
        fpath = os.path.join(folder, fname)
        img = Image.open(fpath)
        images.append(img)

    # boring matplotlib code to show images
    fig, axes = plt.subplots(1, len(images), figsize=(8, 3))
    fig.suptitle(species, fontsize=14)
    for ax, img in zip(axes, images):
        ax.imshow(img)
        ax.axis("off")

    plt.show()


We will pick a pretrained image classifier, ResNet18, and fine-tune it for these plant images. The pre-trained weights are downloaded automatically the first time we use them (an additional download separate from our small dataset).

To use a model we need to know:
- the architecture: number of layers, types of layers, activation functions, etc;
- the weights (unless we want to train from scratch);
- the preprocessing that should be applied to the input images.

The ResNet18 [documentation](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html) has more details. The penguin example above remains on CPU: the dataset and model are too small to benefit from moving every batch to a GPU. We introduce the device switch here, where image processing is more substantial.


In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# Colab default: select Runtime > Change runtime type > GPU first.
# For a CPU run, comment out the next line and uncomment the one below it.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print("Image model device:", device)

weights = ResNet18_Weights.DEFAULT
weights


Notice the name of the weights includes `IMAGENET1K`.

Just as with all the ML techniques we have seen, the data has to match what the model expects. If we have try to do inference with the features in different order / scaling / encoding than we have trained, it will misbehave. The same is true with image data, so the model stores the set of transformations that we have to do to the image to get it into the right format. We could probably track this information down if we had to, but it saves a lot of time to get it straight from the model itself:

In [ ]:
# resnet wants the images to be cropped to 224 x 224 pixels and have normalised RGB values
transform = weights.transforms()  
transform

Now we can do some set up to get the data arranged. Remember from the penguins example that we don't just pass in rows of data in `pytorch`. Everything has to first go through a dataset, then a dataloader. 

In the image-specific package `torchvision` there's an `ImageFolder` dataset type that assumes the images are in labelled folders, so we can just use that:

In [ ]:
# now that we know how the images should be preprocessed, we can point at the images folder:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import random


dataset = datasets.ImageFolder(data_dir, transform=transform)
dataset

Here we will add a bit of custom logic - let's start with just 10 images per species. This is normal Python code, apart from the `Subset` class which is a pytorch helper class for exactly this job:

In [ ]:
# Choose the same 10 images per class each time, whatever file order we see.
MAX_PER_CLASS = 10
labels_list = [label for _, label in dataset.samples]
selected_indices = []
rng = random.Random(42)
for class_idx in range(len(dataset.classes)):
    cls_idxs = [i for i, label in enumerate(labels_list) if label == class_idx]
    selected_indices.extend(rng.sample(cls_idxs, MAX_PER_CLASS))

subset = Subset(dataset, selected_indices)
len(subset)


Once we start the training, all of our outputs will be numerical lables, so it will be important to keep this list of corresponding species names somewhere

In [ ]:
dataset.classes

We will keep eight images of each species for training and two for validation. This gives every class a place in both sets, which a plain random split cannot guarantee with such a small subset. Images of the same original plant at different growth stages could still appear on both sides; the validation accuracy is only a classroom illustration, not a claim of generalisation to new plants.


In [ ]:
train_indices, val_indices = [], []
for class_idx in range(len(dataset.classes)):
    these = [i for i in selected_indices if labels_list[i] == class_idx]
    train_indices.extend(these[:8])
    val_indices.extend(these[8:])

train_ds = Subset(dataset, train_indices)
val_ds = Subset(dataset, val_indices)
len(train_ds), len(val_ds)


The input transformation (crop, resizing and normalisation) otherwise runs again every epoch. There are only 120 images here, so we can apply it **once**, store the resulting tensors in memory, and make loaders from those. This is convenient for this little fixed dataset; for a large dataset or changing random augmentations, we would normally keep transformations in the loader instead.

The cached images stay in normal computer memory. The loader's `pin_memory` option, when a CUDA GPU is selected, can make transfers to the GPU more efficient. There are no worker processes to configure, which also keeps this cell portable to Windows and macOS.


In [ ]:
from torch.utils.data import TensorDataset

train_examples = [train_ds[i] for i in range(len(train_ds))]
val_examples = [val_ds[i] for i in range(len(val_ds))]

train_ds = TensorDataset(
    torch.stack([image for image, label in train_examples]),
    torch.tensor([label for image, label in train_examples]),
)
val_ds = TensorDataset(
    torch.stack([image for image, label in val_examples]),
    torch.tensor([label for image, label in val_examples]),
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=32,
                        pin_memory=(device.type == "cuda"))
train_loader, val_loader


Now that the data are ready to go, we can take a look at the model. We will load it straight with the pretrained weights. Just as with our model we can look at the representation:

In [ ]:
# let's take a look at the model
# the architecture is obviously way more complicated than ours
# lots of convolutional layers
# note the name/sizes of the last layer
model = resnet18(weights=weights)
model

Becuase this model has been set up with a bit more care than ours, it has names for all the layers, so we can address them by name. In particular we can refer to the final layer, called `fc` for fully connected:

In [ ]:
# we can also address this last layer by name
model.fc

This is the one that we want to fine tune. We have to do two things. 

Firstly, notice the size - since this model was originally trained on a dataset with 1000 classes, that's the number of output neurons it has. We need to change it to match the number of outputs in our dataset, which we can do just by overwriting it:

In [ ]:
model.fc = nn.Linear(
    model.fc.in_features,   # keep the same number of inputs....
    len(dataset.classes) # ...but set the number of outputs to match our dataset
)
model = model.to(device)
model

Only the final layer needs new trainable weights. The rest of the pretrained network will act as a fixed feature extractor. The model was moved to `device` above; we will move each batch of images and labels to the **same** device below. With a CUDA GPU selected, `pin_memory` and `non_blocking=True` can help transfer the cached CPU batches.

For a frozen pretrained backbone we keep the network in evaluation mode even when training the final layer: batch-normalisation running statistics should not change. This does **not** prevent gradients through the new final layer.


In [ ]:
[n for n,p in model.named_parameters()]

We want to make sure that we will only adjust the final ones, which we can figure out from the name:

In [ ]:
# we will set all of them to be not changed (requires_grad) except the final layer
# this tells pytorch not to calculate gradients for all the other weights
for name, param in model.named_parameters():
    param.requires_grad = name.startswith("fc")

Our training loop is similar to the penguin one. To keep a classroom run manageable, use 30 epochs with CUDA or 8 on CPU by default; feel free to experiment with these numbers. The model's pretrained layers remain frozen, so we only train the new final layer. The progress bar displays the validation accuracy, but do not interpret results from only two validation images per class as a robust estimate.


In [ ]:
from tqdm.auto import tqdm

# New weights in the final layer; the rest of ResNet is frozen.
model.fc.reset_parameters()
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.fc.parameters(), lr=1e-3)

# Change these if you have more time. CPU fallback remains usable.
num_epochs = 30 if device.type == "cuda" else 8
results = []

# Keep pretrained batch-normalisation statistics fixed during training.
model.eval()
for epoch in tqdm(range(1, num_epochs + 1)):
    for images, labels in train_loader:
        images = images.to(device, non_blocking=(device.type == "cuda"))
        labels = labels.to(device, non_blocking=(device.type == "cuda"))
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=(device.type == "cuda"))
            labels = labels.to(device, non_blocking=(device.type == "cuda"))
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    results.append((epoch, val_acc))
    print(f"Epoch {epoch:02d} | validation accuracy {val_acc:.3f}")


With 12 classes, guessing uniformly at random would get roughly 1/12 correct on average. Compare your accuracy above with that baseline, remembering that the validation set has only 24 images. We have saved the accuracies in `results` so you can plot them later.

Let's pull three images randomly from the validation set and see what our trained model makes of them.


In [ ]:
model.eval()  # put the model in evaluation mode 

# boring matplotlib stuff
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Randomly choose 3 distinct indices from the validation subset
indices = random.sample(range(len(val_ds)), 3)

for ax, idx in zip(axes, indices):

    # get the image and the label as a number
    img, label = val_ds[idx]

    # the only thing we need to do with the label is to use it to look up the true label string
    true_label = dataset.classes[label]

    # turn off the gradient as we are not doing training
    with torch.no_grad():
        
        # Models expect a batch dimension: (N, C, H, W). `unsqueeze(0)` adds N=1.
        # We also move the input to the same device as the model.

        # run the model on the image; because it expects multiple images we have to package it with unsqueeze
        logits = model(img.unsqueeze(0).to(device))
        
        # easier for humans to look at probabilities so we softmax
        probs = torch.softmax(logits, dim=1)
        
        # the results are as a batch, so drop the extra dimension
        probs = probs.squeeze(0).cpu()  # move probabilities back for printing
        
        # Predicted class index is the argmax of probabilities (equivalently logits)
        predicted_class_number = int(probs.argmax())
        predicted_class = dataset.classes[predicted_class_number]

    
    # some boring manipulation to display the image
    # the order of the arrays is different between pytorch and matplotlib so reorder them
    npimg = img.permute(1, 2, 0).numpy()
    
    # and rescale the RBG values so they can be displayed
    npimg = (npimg - npimg.min()) / (npimg.max() - npimg.min() + 1e-8)

    # boring matplotlib code; show the image and predictions
    ax.imshow(npimg)
    ax.axis("off")
    ax.set_title(f"True: {true_label} Pred: {predicted_class}")

    print(f"Image idx {idx} — True: {true_label} | Pred: {predicted_class}")
    for class_idx, class_name in enumerate(dataset.classes):
        print(f"  {class_name:>20}: {probs[class_idx]:.3f}")

plt.tight_layout()
plt.show()


Try switching the device assignment above and rerunning the **image section** to compare CPU and GPU training times. Selecting a GPU runtime alone does not accelerate CPU tensors: the model and image batches must be on the same device. The little penguin example before this section intentionally stays on CPU.
